<a href="https://colab.research.google.com/github/KesteHarshada87/Gen-AI-Practical/blob/main/Expt06(ReAct).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.7 MB/s eta 0:00:00


In [2]:
class SimpleEnvironment:
    def __init__(self):
        self.position = 0
        self.goal = 5

    def get_state(self):
        return f"You are at position {self.position}. Goal is at {self.goal}."

    def step(self, action):
        if action == "right":
            self.position += 1
        elif action == "left":
            self.position -= 1

        # Boundaries
        self.position = max(0, min(self.position, 5))

        done = self.position == self.goal

        return self.get_state(), done

In [7]:
from groq import Groq

class ReActAgent:
    def __init__(self, api_key):
        self.client = Groq(api_key=api_key)
        self.memory = []

    def reason_and_act(self, state):
        prompt = f"""
You are an intelligent agent.

Environment:
{state}

Previous steps:
{self.memory}

Decide next action.

Available actions: left, right

Respond in this format:
Thought: <your reasoning>
Action: <left or right>
"""

        response = self.client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )

        output = response.choices[0].message.content

        print("\nLLM Output:\n", output)

        # Extract action
        action = "right" if "right" in output.lower() else "left"

        return action, output

In [8]:
def run_agent():
    env = SimpleEnvironment()
    agent = ReActAgent(api_key="*************")

    state = env.get_state()

    for step in range(10):
        print(f"\nStep {step + 1}")
        print("State:", state)

        action, reasoning = agent.reason_and_act(state)

        # Store memory
        agent.memory.append(reasoning)

        # Act in environment
        state, done = env.step(action)

        print("Action Taken:", action)

        if done:
            print("\n✅ Goal Reached!")
            break

run_agent()


Step 1
State: You are at position 0. Goal is at 5.

LLM Output:
 Thought: Since my current position is 0 and the goal is at 5, to progress towards the goal, I must take a right action as both left and right options would move me further toward our target location.
Action Taken: right

Step 2
State: You are at position 1. Goal is at 5.

LLM Output:
 Thought: Since my current position is 1 and the goal is at 5, taking the right action will bring me closer to the goal. If I take the left action, I will move in the opposite direction, which is not beneficial. Therefore, taking the right action should allow me to progress towards the goal.

Action: right
Action Taken: right

Step 3
State: You are at position 2. Goal is at 5.

LLM Output:
 Thought: My current position is 2, and the goal remains at 5. Since taking the right action previously brought me closer to the goal, taking the right action again should allow me to continue progressing towards my target location.

Action: right
Action T